L1Loss

In [ ]:
import torch
from torch import nn

input = torch.Tensor([1,2,3])
target = torch.Tensor([1,2,5])

input = torch.reshape(input,(1,1,1,3))
target = torch.reshape(target,(1,1,1,3))

loss = nn.L1Loss()
res = loss(input,target)
print(res)

tensor(0.6667)


MSE

In [ ]:
import torch
from torch import nn

input = torch.Tensor([1,2,3])
target = torch.Tensor([1,2,5])

input = torch.reshape(input,(1,1,1,3))
target = torch.reshape(target,(1,1,1,3))

loss = nn.MSELoss()
res = loss(input,target)
print(res)

tensor(1.3333)


CrossEntropyLoss

e.g.1 三分类

In [ ]:
import torch
from torch import nn

# 1. 准备模型的原始输出得分 (Logits)，输入 x 对应哈士奇分类案例: [人类得分 0.1, 狗得分 0.2, 猫得分 0.3]；形状 (Shape): 一维张量
x = torch.tensor([0.1, 0.2, 0.3], dtype=torch.float32)

# 2. 准备真实标签 y (Target)
'''nn.CrossEntropyLoss 要求真实标签必须是「长整型 (Long/Int64)」，若使用大写字母开头的 torch.Tensor([1])，PyTorch 会默认创建一个“浮点型（Float）”张量，应使用小写的 torch.tensor，并指定数据类型为 dtype=torch.long'''
y = torch.tensor([1], dtype=torch.long)  # 标签为 1，代表真实类别是“狗（Dog）”

# 3. 维度对齐：将输入 x 重构为 2D 矩阵
'''交叉熵损失函数规定，输入模型的分数必须是二维的 [Minibatch, Num_classes]，案例里我们有 1 个样本，3 个类别，因此必须重塑为 (1, 3) 的二维矩阵，转换后x 形状变为: [1, 3]'''
x = torch.reshape(x, (1, 3))

# 4. 实例化多分类交叉熵损失函数
loss_fn = nn.CrossEntropyLoss()

# 5. 计算损失值：传入 [1, 3] 维度的预测值 x，和 [1] 维度的真实标签 y
res = loss_fn(x, y)

# 6. 打印输出最终结果：预期输出为: tensor(1.1019) ，应与刚才手算推导的 Loss 结果 1.1019 吻合
print("计算出的交叉熵损失值为:", res.item())

计算出的交叉熵损失值为: 1.1019428968429565


完整版

In [ ]:
import torch
import torch.nn as nn

# 1. 准备初始数据 (以哈士奇案例为例)
# 类别映射：0代表人类(Person), 1代表狗(Dog), 2代表猫(Cat)
# 哈士奇是一只狗，所以真实类别 target = 1

# 模拟神经网络输出的原始得分 (Logits，未经过 Softmax 激活)
# 形状必须满足 2D 矩阵规范: [Batch_size, Num_classes] ──> [1, 3]
logits = torch.tensor([[0.1, 0.2, 0.3]], dtype=torch.float32)

# 真实标签 Y_target
# 形状为一维长整型张量: [1]，类型必须是 torch.long
Y_target = torch.tensor([1], dtype=torch.long)


# 2. 训练阶段：计算高精度的交叉熵损失 (Loss)
# 实例化 PyTorch 官方的多分类交叉熵损失函数
# 此时底层会自动执行高稳定的 Log-Sum-Exp 转换，彻底避免计算过程中的数值溢出风险
criterion = nn.CrossEntropyLoss()

loss = criterion(logits, Y_target)

print("===== 训练/评估阶段 =====")
# 打印算出来的 Loss 值（应该约为 1.1019，这与之前手算公式得到的结果完全一致）
print(f"当前计算出的交叉熵损失值 (Loss) 为: {loss.item():.4f}\n")


# 3. 预测阶段：手动在外部进行概率恢复 (Probability Recovery)
print("===== 预测/推理阶段 =====")

# 据此实现：因为训练时模型最末端没有加激活函数（输出的是 Logits）
# 此时如果直接看 logits [0.1, 0.2, 0.3]，其数值没有概率意义，且和不为 1
# 因此在预测阶段，我们需要手动在外部套上对应的 softmax 函数，将其恢复为 0~1 的真实概率！ [1.1.2, 1.2.3]
# dim=1 表示在第 1 维度（即类别数量维度）上进行归一化 [1.2.3]
probabilities = torch.softmax(logits, dim=1)

# 打印恢复出来的真实概率
print("模型预测当前样本在 3 个类别上的概率分别为:")
print(f"  -> 是“人类(0)”的概率: {probabilities[0][0].item() * 100:.2f}%") # 约 30.06%
print(f"  -> 是“狗(1)”的概率:   {probabilities[0][1].item() * 100:.2f}%") # 约 33.22%
print(f"  -> 是“猫(2)”的概率:   {probabilities[0][2].item() * 100:.2f}%") # 约 36.72%

# 验证概率之和是否完美等于 100% (1.0)
print(f"  -> 3个通道预测概率的总和为: {torch.sum(probabilities).item():.4f}") # 1.0000

# 4. 获取最终的决策分类索引 (取概率最大的那个作为预测结论)
predicted_class = torch.argmax(probabilities, dim=1).item()
class_names = {0: "人类", 1: "狗", 2: "猫"}
print(f"\n模型最终判定：这张哈士奇图片属于 ──> 【{class_names[predicted_class]}】")

===== 训练/评估阶段 =====
当前计算出的交叉熵损失值 (Loss) 为: 1.1019

===== 预测/推理阶段 =====
模型预测当前样本在 3 个类别上的概率分别为:
  -> 是“人类(0)”的概率: 30.06%
  -> 是“狗(1)”的概率:   33.22%
  -> 是“猫(2)”的概率:   36.72%
  -> 3个通道预测概率的总和为: 1.0000

模型最终判定：这张哈士奇图片属于 ──> 【猫】


e.g.2 CIFAR10

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
import torchvision
from torch.nn import Conv2d, MaxPool2d, Flatten, Linear
from torch.utils.tensorboard import SummaryWriter

dataset = torchvision.datasets.CIFAR10("../data", train=False, transform=torchvision.transforms.ToTensor(),
                                       download=True)
dataloader = DataLoader(dataset, batch_size=1)

class model(nn.Module):
    def __init__(self):
        super(model, self).__init__()
        self.model1 = nn.Sequential(
            Conv2d(in_channels=3, out_channels=32, kernel_size=5, stride=1, padding=2),
            MaxPool2d(2),
            Conv2d(in_channels=32, out_channels=32,kernel_size=5, stride=1, padding=2),
            MaxPool2d(2),
            Conv2d(in_channels=32, out_channels=64, kernel_size=5, stride=1, padding=2),
            MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(1024, 64),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        x = self.model1(x)
        return x

loss = nn.CrossEntropyLoss()

test = model()

for data in dataloader:
  imgs,targets = data
  outputs=test(imgs)
  loss_result = loss(outputs,targets)
  print(loss_result)

100%|██████████| 170M/170M [00:01<00:00, 90.7MB/s]


流式输出内容被截断，只能显示最后 5000 行内容。
tensor(2.4129, grad_fn=<NllLossBackward0>)
tensor(2.1863, grad_fn=<NllLossBackward0>)
tensor(2.2574, grad_fn=<NllLossBackward0>)
tensor(2.3290, grad_fn=<NllLossBackward0>)
tensor(2.3276, grad_fn=<NllLossBackward0>)
tensor(2.3819, grad_fn=<NllLossBackward0>)
tensor(2.4066, grad_fn=<NllLossBackward0>)
tensor(2.3055, grad_fn=<NllLossBackward0>)
tensor(2.3064, grad_fn=<NllLossBackward0>)
tensor(2.4048, grad_fn=<NllLossBackward0>)
tensor(2.4105, grad_fn=<NllLossBackward0>)
tensor(2.3102, grad_fn=<NllLossBackward0>)
tensor(2.1804, grad_fn=<NllLossBackward0>)
tensor(2.2178, grad_fn=<NllLossBackward0>)
tensor(2.4332, grad_fn=<NllLossBackward0>)
tensor(2.3678, grad_fn=<NllLossBackward0>)
tensor(2.1882, grad_fn=<NllLossBackward0>)
tensor(2.4138, grad_fn=<NllLossBackward0>)
tensor(2.3294, grad_fn=<NllLossBackward0>)
tensor(2.4059, grad_fn=<NllLossBackward0>)
tensor(2.2747, grad_fn=<NllLossBackward0>)
tensor(2.2180, grad_fn=<NllLossBackward0>)
tensor(2.3538, grad_fn=<Nll

add backward

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
import torchvision
from torch.nn import Conv2d, MaxPool2d, Flatten, Linear
from torch.utils.tensorboard import SummaryWriter

dataset = torchvision.datasets.CIFAR10("../data", train=False, transform=torchvision.transforms.ToTensor(),
                                       download=True)
dataloader = DataLoader(dataset, batch_size=1)

class model(nn.Module):
    def __init__(self):
        super(model, self).__init__()
        self.model1 = nn.Sequential(
            Conv2d(in_channels=3, out_channels=32, kernel_size=5, stride=1, padding=2),
            MaxPool2d(2),
            Conv2d(in_channels=32, out_channels=32,kernel_size=5, stride=1, padding=2),
            MaxPool2d(2),
            Conv2d(in_channels=32, out_channels=64, kernel_size=5, stride=1, padding=2),
            MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(1024, 64),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        x = self.model1(x)
        return x

loss = nn.CrossEntropyLoss()

test = model()

for data in dataloader:
  imgs,targets = data
  outputs=test(imgs)
  loss_result = loss(outputs,targets)
  loss_result.backward()